# Probability Distributions for Quantitative Finance & Risk
### Theory, full derivations, and **live interactive plots** (drag the sliders, the graph updates instantly)

**How to use this notebook**
1. Run the first code cell (setup). Needs `numpy, scipy, matplotlib, ipywidgets`
   (`pip install numpy scipy matplotlib ipywidgets`).
2. Works in **JupyterLab, Jupyter Notebook, VS Code and Google Colab**.
3. Every distribution has: **meaning -> formula -> derivations (mean, variance, ...) -> where it is used -> live plot**.
4. Drag any slider in a plot cell. The plot redraws live. No need to edit code or re-run.

**Notation:** $\mu$ = mean, $\sigma^2$ = variance, skewness $=E[(X-\mu)^3]/\sigma^3$, kurtosis $=E[(X-\mu)^4]/\sigma^4$, excess kurtosis = kurtosis $-3$.

**Contents**
- Part 0: Maths toolkit (moments, MGF, cumulants, Gamma/Beta functions, series)
- Part A: Discrete: Bernoulli, Binomial, Geometric, Poisson
- Part B: Continuous standard: Uniform, Normal, Lognormal, Exponential, Gamma, Chi-square, Student-t, F, Beta, Weibull, Laplace
- Part C: Fat-tail family: Cauchy, Pareto, Levy, alpha-stable
- Part D: Extreme value: GEV, GPD
- Part E: Multivariate Normal (and why it underestimates joint crashes)
- Cheat sheet and tail comparison

In [1]:
# ---------- SETUP: run this cell first ----------
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as st
from scipy import special
from ipywidgets import interact, interactive, fixed, widgets

# Live sliders: if plots do not refresh in your frontend, run:  %matplotlib inline
%matplotlib inline
plt.rcParams.update({'figure.figsize': (11, 4), 'axes.grid': True, 'grid.alpha': .3, 'font.size': 11})

def stats_str(d):
    """mean / variance / skew / excess kurtosis of a frozen scipy distribution as text"""
    m, v, s, k = d.stats(moments='mvsk')
    f = lambda x: 'undef' if not np.isfinite(float(x)) else f'{float(x):.3f}'
    return f'mean={f(m)}  var={f(v)}  skew={f(s)}  ex.kurt={f(k)}'

def stem(ax, ks, pm, color='C0', label=None):
    ax.vlines(ks, 0, pm, color=color, lw=4, alpha=.75)
    ax.plot(ks, pm, 'o', color=color, label=label)
print('Setup done.')

Setup done.


---
# PART 0: Maths toolkit (used in every derivation below)

### 0.1 Expectation and variance
For a discrete variable: $E[g(X)]=\sum_x g(x)P(X=x)$. For a continuous one: $E[g(X)]=\int g(x)f(x)\,dx$.

* **Mean** $\mu=E[X]$ (centre of mass).
* **Variance** $\sigma^2=E[(X-\mu)^2]$ (average squared distance from the mean).

**Shortcut identity (used everywhere):**
$$\operatorname{Var}(X)=E[(X-\mu)^2]=E[X^2]-2\mu E[X]+\mu^2=E[X^2]-\mu^2 .$$

**Linearity:** $E[aX+bY]=aE[X]+bE[Y]$ always. **Variance of sums:** $\operatorname{Var}(X+Y)=\operatorname{Var}X+\operatorname{Var}Y+2\operatorname{Cov}(X,Y)$, and the covariance term is $0$ if $X,Y$ are independent.
Also $\operatorname{Var}(aX+b)=a^2\operatorname{Var}(X)$.

### 0.2 Factorial moment trick (for discrete distributions)
Often $E[X(X-1)]$ is easier than $E[X^2]$. Then
$$\operatorname{Var}(X)=E[X(X-1)]+E[X]-E[X]^2 .$$

### 0.3 Moment generating function (MGF) and cumulants
$$M_X(t)=E[e^{tX}],\qquad E[X^n]=M_X^{(n)}(0).$$
Reason: $e^{tX}=\sum_n t^nX^n/n!$, so $M_X(t)=\sum_n E[X^n]t^n/n!$. Differentiating $n$ times at $0$ leaves $E[X^n]$.

If $X,Y$ independent: $M_{X+Y}(t)=E[e^{tX}e^{tY}]=M_X(t)M_Y(t)$. The MGF turns sums into products.

**Cumulant generating function** $K(t)=\ln M(t)$. Its derivatives at 0 are cumulants $\kappa_n$:
$\kappa_1=\mu$, $\kappa_2=\sigma^2$, $\kappa_3=E[(X-\mu)^3]$, $\kappa_4=E[(X-\mu)^4]-3\sigma^4$.
Hence skewness $=\kappa_3/\sigma^3$ and **excess kurtosis $=\kappa_4/\sigma^4$**. That is why "excess" kurtosis is zero for the Normal: all cumulants above 2 vanish.

When the MGF does not exist (fat tails), we use the **characteristic function** $\varphi(t)=E[e^{itX}]$, which always exists.

### 0.4 Useful series and integrals
* **Geometric series:** $\sum_{k=0}^\infty q^k=\dfrac1{1-q}$ for $|q|<1$.
  Differentiating: $\sum_{k\ge1}kq^{k-1}=\dfrac1{(1-q)^2}$, and again: $\sum_{k\ge2}k(k-1)q^{k-2}=\dfrac{2}{(1-q)^3}$.
* **Exponential series:** $e^\lambda=\sum_{k\ge0}\lambda^k/k!$.
* **Binomial theorem:** $(a+b)^n=\sum_k\binom nk a^kb^{n-k}$.
* **Gamma function:** $\Gamma(z)=\int_0^\infty t^{z-1}e^{-t}dt$.
  Integration by parts ($u=t^{z}$, $dv=e^{-t}dt$) gives $\Gamma(z+1)=z\Gamma(z)$. Since $\Gamma(1)=\int_0^\infty e^{-t}dt=1$, we get $\Gamma(n)=(n-1)!$. Also $\Gamma(\tfrac12)=\sqrt\pi$.
* **Beta function:** $B(a,b)=\int_0^1 t^{a-1}(1-t)^{b-1}dt=\dfrac{\Gamma(a)\Gamma(b)}{\Gamma(a+b)}$.
* **Gaussian integral:** $\int_{-\infty}^\infty e^{-x^2/2}dx=\sqrt{2\pi}$ (proved in the Normal section).

### 0.5 Inverse-transform sampling
If $U\sim U(0,1)$ and $F$ is a CDF, then $X=F^{-1}(U)$ has CDF $F$, because $P(F^{-1}(U)\le x)=P(U\le F(x))=F(x)$. This is how a computer produces any distribution from uniform random numbers.

---
# PART A: DISCRETE DISTRIBUTIONS (countable outcomes)

---
## 1. Bernoulli($p$): one yes/no trial

### Meaning
The simplest random experiment: one trial, two outcomes. "Success" (1) with probability $p$, "failure" (0) with probability $1-p$. Every other discrete distribution in this part is built from Bernoulli trials.

### Formula
$$P(X=1)=p,\quad P(X=0)=1-p,\qquad\text{compactly } P(X=x)=p^x(1-p)^{1-x},\ x\in\{0,1\}.$$

### Derivation of mean and variance
$$E[X]=0\cdot(1-p)+1\cdot p=p.$$
Since $X\in\{0,1\}$, $X^2=X$, so $E[X^2]=p$. Therefore
$$\operatorname{Var}(X)=E[X^2]-E[X]^2=p-p^2=p(1-p).$$

### Where variance is maximal
Let $g(p)=p-p^2$. Then $g'(p)=1-2p=0\Rightarrow p=\tfrac12$, and $g''=-2<0$ so it is a maximum with $g(\tfrac12)=\tfrac14$. Intuition: at $p=\tfrac12$ you are maximally uncertain; at $p=0$ or $1$ the outcome is certain and variance is $0$.

Skewness $=\dfrac{1-2p}{\sqrt{p(1-p)}}$ (positive for $p<\tfrac12$).

### Where it is used (finance)
* Loan default indicator (1 = default, 0 = no default).
* Up-day vs down-day indicator.
* **VaR breach indicator:** 1 if loss on a day exceeds VaR. For a 99% VaR, $p=0.01$.
* Any binary event: merger happens or not, option expires in the money or not.

In [ ]:
def plot_bernoulli(p=0.30):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].bar([0, 1], [1-p, p], color=['#c0392b', '#27ae60'], width=.5)
    ax[0].set_xticks([0, 1]); ax[0].set_ylim(0, 1)
    ax[0].set_title(f'PMF   mean={p:.2f}   var={p*(1-p):.3f}')
    pp = np.linspace(0, 1, 300)
    ax[1].plot(pp, pp*(1-pp)); ax[1].plot(p, p*(1-p), 'ro', ms=9)
    ax[1].set_title('Variance p(1-p): maximal (0.25) at p=0.5'); ax[1].set_xlabel('p')
    plt.show()
interact(plot_bernoulli, p=(0.0, 1.0, 0.01));

interactive(children=(FloatSlider(value=0.3, description='p', max=1.0, step=0.01), Output()), _dom_classes=('w…

---
## 2. Binomial($n,p$): number of successes in $n$ independent Bernoulli trials

### Meaning
Repeat the same yes/no experiment $n$ times independently and count the successes. Example: out of 1000 loans, how many default?

### Derivation of the PMF
A particular sequence with exactly $k$ successes (e.g. S S F S F ...) has probability $p^k(1-p)^{n-k}$ by independence. The number of different sequences containing exactly $k$ successes is $\binom nk$ (choose which $k$ of the $n$ positions are successes). Adding them up:
$$P(X=k)=\binom nk p^k(1-p)^{n-k},\qquad k=0,\dots,n.$$
It sums to 1 by the binomial theorem: $\sum_k\binom nk p^k(1-p)^{n-k}=(p+1-p)^n=1$.

### Derivation of mean and variance (easy way: sum of Bernoullis)
$X=B_1+\dots+B_n$ with $B_i\sim$ Bernoulli($p$) iid.
$$E[X]=\sum E[B_i]=np.$$
By independence the covariances vanish:
$$\operatorname{Var}(X)=\sum\operatorname{Var}(B_i)=np(1-p).$$

### Direct derivation of the mean (to see the algebra)
Use $k\binom nk=n\binom{n-1}{k-1}$:
$$E[X]=\sum_{k=1}^nk\binom nkp^k q^{n-k}=np\sum_{k=1}^n\binom{n-1}{k-1}p^{k-1}q^{n-k}=np\,(p+q)^{n-1}=np\quad(q=1-p).$$

### Effect of changing $n$ and $p$
* **Relative spread** $\sigma/\mu=\sqrt{\dfrac{np(1-p)}{n^2p^2}}=\sqrt{\dfrac{1-p}{np}}\propto\dfrac1{\sqrt n}$ shrinks as $n$ grows (law of large numbers).
* Skewness $=\dfrac{1-2p}{\sqrt{np(1-p)}}$: right-skewed if $p<\tfrac12$, left-skewed if $p>\tfrac12$, symmetric at $\tfrac12$.

### Limit 1: Normal (De Moivre-Laplace)
For large $n$, $X\approx N(np,\,np(1-p))$. Reason: $X$ is a sum of $n$ iid variables, so the Central Limit Theorem applies. (Use continuity correction: $P(X\le k)\approx\Phi\big(\frac{k+0.5-np}{\sqrt{npq}}\big)$.)

### Limit 2: Poisson (law of rare events)
Let $n\to\infty$, $p\to0$ with $np=\lambda$ fixed:
$$\binom nk p^k(1-p)^{n-k}=\frac{n(n-1)\cdots(n-k+1)}{k!}\frac{\lambda^k}{n^k}\Big(1-\frac\lambda n\Big)^{n-k}
=\underbrace{\frac{n}{n}\cdots\frac{n-k+1}{n}}_{\to1}\ \frac{\lambda^k}{k!}\ \underbrace{\Big(1-\frac\lambda n\Big)^{n}}_{\to e^{-\lambda}}\ \underbrace{\Big(1-\frac\lambda n\Big)^{-k}}_{\to1}\to\frac{e^{-\lambda}\lambda^k}{k!}.$$

### Where it is used
* Number of defaults in a loan pool (if independent with same PD).
* **VaR backtesting:** with 99% VaR over 250 days, breaches $\sim$ Bin(250, 0.01), expected $=2.5$. The **Basel traffic-light** test counts breaches: 0-4 green, 5-9 yellow, 10+ red. Probabilities come directly from this formula.
* Binomial option-pricing tree (number of up-moves).

In [3]:
def plot_binomial(n=50, p=0.30, normal_approx=True, poisson_approx=False):
    ks = np.arange(0, n+1); d = st.binom(n, p)
    fig, ax = plt.subplots(figsize=(11, 4))
    stem(ax, ks, d.pmf(ks), 'C0', 'Binomial PMF')
    x = np.linspace(0, n, 400)
    if normal_approx: ax.plot(x, st.norm(n*p, np.sqrt(n*p*(1-p))).pdf(x), 'r-', label='Normal approx')
    if poisson_approx: ax.plot(ks, st.poisson(n*p).pmf(ks), 'g^--', alpha=.7, label='Poisson approx')
    lo, hi = d.ppf(.0005), d.ppf(.9995); ax.set_xlim(max(lo-2, -1), hi+2)
    ax.set_title(f'Binomial(n={n}, p={p:.2f})   {stats_str(d)}   rel.spread={np.sqrt((1-p)/(n*p)) if p>0 else 0:.3f}')
    ax.legend(); plt.show()
interact(plot_binomial, n=(1, 500, 1), p=(0.01, 0.99, 0.01));

interactive(children=(IntSlider(value=50, description='n', max=500, min=1), FloatSlider(value=0.3, description…

In [4]:
# Basel VaR backtest: how likely are k or more breaches in a year?
def basel_backtest(n_days=250, var_level=0.99, breaches_seen=5):
    p = 1 - var_level; d = st.binom(n_days, p)
    ks = np.arange(0, 31)
    zone = np.where(ks <= 4, 'green', np.where(ks <= 9, 'gold', 'red'))
    fig, ax = plt.subplots(figsize=(11, 3.8))
    ax.bar(ks, d.pmf(ks), color=zone, edgecolor='k', alpha=.8)
    ax.axvline(breaches_seen, color='b', ls='--', label=f'observed = {breaches_seen}')
    ax.set_title(f'Bin({n_days}, {p:.3f}); expected={n_days*p:.2f};  P(X >= {breaches_seen}) = {d.sf(breaches_seen-1):.4f}')
    ax.legend(); plt.show()
interact(basel_backtest, n_days=(100, 1000, 10), var_level=(0.90, 0.999, 0.001), breaches_seen=(0, 30, 1));

interactive(children=(IntSlider(value=250, description='n_days', max=1000, min=100, step=10), FloatSlider(valu…

---
## 3. Geometric($p$): number of trials until the first success

### Meaning
Keep repeating independent Bernoulli($p$) trials. $X$ = the trial number on which the first success occurs ($k=1,2,3,\dots$).

### Derivation of the PMF
First success at trial $k$ means $k-1$ failures then one success:
$$P(X=k)=(1-p)^{k-1}p.$$
Check: $\sum_{k\ge1}q^{k-1}p=p\cdot\frac1{1-q}=1$.

### Derivation of the mean
With $q=1-p$:
$$E[X]=\sum_{k\ge1}k\,q^{k-1}p=p\sum_{k\ge1}kq^{k-1}=p\cdot\frac1{(1-q)^2}=\frac p{p^2}=\frac1p.$$
(Used $\sum kq^{k-1}=1/(1-q)^2$ from Part 0.)

### Derivation of the variance (factorial-moment trick)
$$E[X(X-1)]=\sum_{k\ge2}k(k-1)q^{k-1}p=pq\sum_{k\ge2}k(k-1)q^{k-2}=pq\cdot\frac2{(1-q)^3}=\frac{2q}{p^2}.$$
Then
$$\operatorname{Var}(X)=E[X(X-1)]+E[X]-E[X]^2=\frac{2q}{p^2}+\frac1p-\frac1{p^2}=\frac{2q+p-1}{p^2}=\frac{2(1-p)+p-1}{p^2}=\frac{1-p}{p^2}.$$

### Memoryless property (proof)
$P(X>m)=q^m$ (first $m$ trials all fail). So
$$P(X>m+n\mid X>m)=\frac{P(X>m+n)}{P(X>m)}=\frac{q^{m+n}}{q^m}=q^n=P(X>n).$$
Past failures tell you nothing: the process "restarts" each trial. (Geometric is the only memoryless discrete distribution.)

### Where it is used
* Days until the first VaR breach (mean $=1/p=100$ days for 99% VaR).
* Discrete-time time-to-default with constant default probability per period.
* Number of attempts until a rare event (first loan default in a stream of loans).

In [ ]:
def plot_geometric(p=0.10, m=10, n=5):
    d = st.geom(p); ks = np.arange(1, int(min(d.ppf(.999), 400))+1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    stem(ax[0], ks, d.pmf(ks)); ax[0].set_title(f'Geometric(p={p:.2f})  {stats_str(d)}')
    # memoryless demo
    lhs = d.sf(m+n)/d.sf(m); rhs = d.sf(n)
    ax[1].bar(['P(X>m+n | X>m)', 'P(X>n)'], [lhs, rhs], color=['C1', 'C2'])
    ax[1].set_title(f'Memoryless: {lhs:.4f} = {rhs:.4f}   (m={m}, n={n})'); ax[1].set_ylim(0, 1)
    plt.show()
interact(plot_geometric, p=(0.01, 0.9, 0.01), m=(0, 50, 1), n=(0, 50, 1));

interactive(children=(FloatSlider(value=0.1, description='p', max=0.9, min=0.01, step=0.01), IntSlider(value=1…

---
## 4. Poisson($\lambda$): number of random events in a fixed interval

### Meaning
Events happen independently at random times with an average rate of $\lambda$ per interval (per year, per month, per second). $X$ counts how many happen. Think "number of jumps in a stock this year", "number of defaults this month", "number of orders in one second".

### Derivation of the PMF (limit of the Binomial)
Split the interval into $n$ tiny pieces; in each piece an event occurs with probability $p=\lambda/n$ (at most one per piece). Then the count is Bin($n,\lambda/n$). Letting $n\to\infty$ (shown in section 2):
$$P(X=k)=\frac{e^{-\lambda}\lambda^k}{k!},\qquad k=0,1,2,\dots$$
Check: $\sum_k e^{-\lambda}\lambda^k/k!=e^{-\lambda}e^{\lambda}=1$.

### Derivation of the mean
$$E[X]=\sum_{k\ge0}k\frac{e^{-\lambda}\lambda^k}{k!}=e^{-\lambda}\sum_{k\ge1}\frac{\lambda^k}{(k-1)!}=\lambda e^{-\lambda}\sum_{k\ge1}\frac{\lambda^{k-1}}{(k-1)!}=\lambda e^{-\lambda}e^{\lambda}=\lambda.$$

### Derivation of the variance
$$E[X(X-1)]=\sum_{k\ge2}k(k-1)\frac{e^{-\lambda}\lambda^k}{k!}=\lambda^2e^{-\lambda}\sum_{k\ge2}\frac{\lambda^{k-2}}{(k-2)!}=\lambda^2.$$
$$\operatorname{Var}(X)=E[X(X-1)]+E[X]-E[X]^2=\lambda^2+\lambda-\lambda^2=\lambda.$$
**Fingerprint: mean = variance.** If real count data has variance $>$ mean (overdispersion), use the Negative Binomial.

### Derivation of skewness and kurtosis (MGF / cumulants)
$$M(t)=\sum_k e^{tk}\frac{e^{-\lambda}\lambda^k}{k!}=e^{-\lambda}\sum_k\frac{(\lambda e^t)^k}{k!}=e^{\lambda(e^t-1)}.$$
$K(t)=\ln M=\lambda(e^t-1)$, so every derivative at 0 is $\lambda$: **all cumulants equal $\lambda$**. So $\kappa_3=\lambda$, $\kappa_4=\lambda$:
$$\text{skewness}=\frac{\lambda}{\lambda^{3/2}}=\frac1{\sqrt\lambda},\qquad\text{excess kurtosis}=\frac{\lambda}{\lambda^2}=\frac1\lambda.$$
Both $\to0$ as $\lambda\to\infty$, so Poisson becomes Normal.

### Sums
If $X_1\sim$Poi($\lambda_1$), $X_2\sim$Poi($\lambda_2$) independent, $M_{X_1+X_2}=e^{(\lambda_1+\lambda_2)(e^t-1)}$, so the sum is Poi($\lambda_1+\lambda_2$).

### Changing $\lambda$
It shifts the mean and widens the spread together. You cannot control them separately (a limitation).

### Where it is used
* **Merton jump-diffusion:** number of jumps $N_T\sim$ Poi($\lambda T$) in $dS/S=\mu dt+\sigma dW+(J-1)dN$.
* Defaults per month (CreditRisk+ model), operational loss event counts, order arrivals in market microstructure.

In [ ]:
def plot_poisson(lam=4.0, normal_approx=True):
    d = st.poisson(lam); ks = np.arange(0, int(d.ppf(.9999))+3)
    fig, ax = plt.subplots(figsize=(11, 4))
    stem(ax, ks, d.pmf(ks), 'C0', 'Poisson PMF')
    if normal_approx:
        x = np.linspace(0, ks[-1], 400); ax.plot(x, st.norm(lam, np.sqrt(lam)).pdf(x), 'r-', label='Normal approx')
    ax.set_title(f'Poisson(lambda={lam:.1f})   {stats_str(d)}   (1/sqrt(lam)={1/np.sqrt(lam):.3f}, 1/lam={1/lam:.3f})')
    ax.legend(); plt.show()
interact(plot_poisson, lam=(0.1, 60, 0.1));

interactive(children=(FloatSlider(value=4.0, description='lam', max=60.0, min=0.1), Checkbox(value=True, descr…

---
# PART B: CONTINUOUS DISTRIBUTIONS ("standard" ones)

For continuous variables the PDF $f(x)$ is a density: $P(a\le X\le b)=\int_a^bf(x)dx$, and $\int f=1$. The CDF is $F(x)=\int_{-\infty}^xf$.

---
## 5. Uniform($a,b$): every value in $[a,b]$ equally likely

### Meaning
Complete ignorance within a range: no value is more likely than another.

### Formula and normalisation
$f(x)=c$ on $[a,b]$. $\int_a^bc\,dx=c(b-a)=1\Rightarrow f(x)=\dfrac1{b-a}$. CDF: $F(x)=\dfrac{x-a}{b-a}$.

### Mean and variance
$$E[X]=\int_a^b\frac{x}{b-a}dx=\frac{b^2-a^2}{2(b-a)}=\frac{a+b}2.$$
$$E[X^2]=\int_a^b\frac{x^2}{b-a}dx=\frac{b^3-a^3}{3(b-a)}=\frac{a^2+ab+b^2}3.$$
$$\operatorname{Var}=\frac{a^2+ab+b^2}3-\frac{(a+b)^2}4=\frac{4a^2+4ab+4b^2-3a^2-6ab-3b^2}{12}=\frac{(b-a)^2}{12}.$$
Skewness $=0$. Excess kurtosis $=-6/5$ (flatter than Normal).

### Why it is the base of all Monte Carlo
Computers generate $U\sim U(0,1)$. **Inverse transform:** $X=F^{-1}(U)$ has CDF $F$ (proof in Part 0.5).
Example, Exponential($\lambda$): $F(x)=1-e^{-\lambda x}$. Solve $u=1-e^{-\lambda x}\Rightarrow x=-\ln(1-u)/\lambda$.

Also: under the null hypothesis, a p-value is $U(0,1)$ (because $P(F(T)\le u)=u$).

### Where it is used
Random number generation, Monte Carlo pricing, priors with no information, randomised tests, copulas (marginals are transformed to uniforms).

In [ ]:
def plot_uniform(a=0.0, b=1.0, lam=1.5, N=3000):
    if b <= a + 0.01: b = a + 0.01
    d = st.uniform(a, b-a); x = np.linspace(a-1, b+1, 500)
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
    ax[0].plot(x, d.pdf(x)); ax[0].fill_between(x, d.pdf(x), alpha=.2); ax[0].set_title('PDF  ' + stats_str(d))
    ax[1].plot(x, d.cdf(x)); ax[1].set_title('CDF')
    rng = np.random.default_rng(1); U = rng.random(N); X = -np.log(1-U)/lam   # inverse transform
    ax[2].hist(X, bins=60, density=True, alpha=.5, label='-ln(1-U)/lam')
    xx = np.linspace(0, X.max(), 300); ax[2].plot(xx, lam*np.exp(-lam*xx), 'r', label='Exponential pdf')
    ax[2].set_title('Inverse-transform sampling'); ax[2].legend()
    plt.tight_layout(); plt.show()
interact(plot_uniform, a=(-3.0, 3.0, 0.1), b=(-2.0, 6.0, 0.1), lam=(0.2, 4.0, 0.1), N=(200, 20000, 200));

interactive(children=(FloatSlider(value=0.0, description='a', max=3.0, min=-3.0), FloatSlider(value=1.0, descr…

---
## 6. Normal $N(\mu,\sigma^2)$: the bell curve

### Meaning
The distribution of a quantity that is the sum of many small independent effects (**Central Limit Theorem**). It is completely determined by two numbers: centre $\mu$ and spread $\sigma$.

### Formula
$$f(x)=\frac1{\sigma\sqrt{2\pi}}\exp\Big(-\frac{(x-\mu)^2}{2\sigma^2}\Big).$$

### Derivation that it integrates to 1 (the Gaussian integral)
Let $I=\int_{-\infty}^\infty e^{-x^2/2}dx$. Then
$$I^2=\int\!\!\int e^{-(x^2+y^2)/2}dx\,dy=\int_0^{2\pi}\!\!\int_0^\infty e^{-r^2/2}r\,dr\,d\theta=2\pi\Big[-e^{-r^2/2}\Big]_0^\infty=2\pi.$$
So $I=\sqrt{2\pi}$. Substituting $z=(x-\mu)/\sigma$ gives $\int f\,dx=\frac1{\sqrt{2\pi}}\int e^{-z^2/2}dz=1$. This is where $\sqrt{2\pi}$ comes from.

### Derivation of mean and variance (standard $Z\sim N(0,1)$, then scale)
Mean: $E[Z]=\int z\frac{e^{-z^2/2}}{\sqrt{2\pi}}dz=0$ because the integrand is odd.
Variance: integrate by parts with $u=z,\ dv=ze^{-z^2/2}dz$ (so $v=-e^{-z^2/2}$):
$$E[Z^2]=\frac1{\sqrt{2\pi}}\Big\{\big[-ze^{-z^2/2}\big]_{-\infty}^\infty+\int e^{-z^2/2}dz\Big\}=\frac1{\sqrt{2\pi}}\{0+\sqrt{2\pi}\}=1.$$
For $X=\mu+\sigma Z$: $E[X]=\mu$, $\operatorname{Var}(X)=\sigma^2\operatorname{Var}(Z)=\sigma^2$.

### MGF (derive by completing the square)
$$M_Z(t)=\int e^{tz}\frac{e^{-z^2/2}}{\sqrt{2\pi}}dz=e^{t^2/2}\int\frac{e^{-(z-t)^2/2}}{\sqrt{2\pi}}dz=e^{t^2/2}.$$
since $tz-z^2/2=-(z-t)^2/2+t^2/2$. For $X$: $M_X(t)=e^{\mu t+\sigma^2t^2/2}$.
Then $K(t)=\mu t+\sigma^2t^2/2$ is a quadratic, so **all cumulants beyond the 2nd are zero**: skewness $=0$, excess kurtosis $=0$ (kurtosis $=3$).
Even moments: $E[Z^4]=3$ (from $M^{(4)}(0)$ of $e^{t^2/2}=1+t^2/2+t^4/8+\dots$; $E[Z^4]=4!/8=3$).

### Sums of independent normals are normal
$M_{X+Y}(t)=e^{(\mu_1+\mu_2)t+(\sigma_1^2+\sigma_2^2)t^2/2}$, i.e. $N(\mu_1+\mu_2,\sigma_1^2+\sigma_2^2)$. Variances add (not standard deviations).

### 68-95-99.7 rule
$P(|X-\mu|<k\sigma)=2\Phi(k)-1$: $k=1\to68.27\%$, $k=2\to95.45\%$, $k=3\to99.73\%$.

### Maximum entropy
Among all densities with given mean and variance, the Normal has the largest entropy $\tfrac12\ln(2\pi e\sigma^2)$: it assumes the least beyond mean and variance.

### Where it is used
Black-Scholes log-returns, parametric (variance-covariance) VaR, CAPM and regression errors, Kalman filters, Brownian motion increments. Great for averages, **bad for extremes** (tails too thin).

In [ ]:
def plot_normal(mu=0.0, sigma=1.0):
    d = st.norm(mu, sigma); x = np.linspace(-12, 12, 1000)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(x, d.pdf(x), 'k')
    for k, c in zip([3, 2, 1], ['#fbd0d0', '#f9e79f', '#aed6f1']):
        m = (x > mu-k*sigma) & (x < mu+k*sigma)
        ax[0].fill_between(x[m], d.pdf(x[m]), color=c, label=f'within {k} sigma: {2*st.norm.cdf(k)-1:.2%}')
    ax[0].set_xlim(-12, 12); ax[0].legend(); ax[0].set_title(f'N({mu:.1f}, {sigma**2:.2f})  ' + stats_str(d))
    ax[1].plot(x, d.cdf(x)); ax[1].set_title('CDF'); ax[1].set_xlim(-12, 12)
    plt.tight_layout(); plt.show()
interact(plot_normal, mu=(-5.0, 5.0, 0.1), sigma=(0.2, 4.0, 0.1));

# Central Limit Theorem demo: sum of n iid variables becomes Normal
def clt_demo(n=1, source='uniform', N=20000):
    rng = np.random.default_rng(0)
    gens = {'uniform': lambda s: rng.random(s), 'exponential': lambda s: rng.exponential(1, s),
            'bernoulli(0.1)': lambda s: (rng.random(s) < .1).astype(float), 'student-t(3)': lambda s: rng.standard_t(3, s)}
    S = gens[source]((N, n)).sum(axis=1); z = (S - S.mean())/S.std()
    fig, ax = plt.subplots(figsize=(11, 3.8)); ax.hist(z, bins=80, density=True, alpha=.5, range=(-5, 5))
    x = np.linspace(-5, 5, 300); ax.plot(x, st.norm.pdf(x), 'r'); ax.set_title(f'CLT: standardised sum of n={n} {source} variables vs N(0,1)')
    plt.show()
interact(clt_demo, n=(1, 200, 1), source=['uniform', 'exponential', 'bernoulli(0.1)', 'student-t(3)'], N=(2000, 50000, 2000));

interactive(children=(FloatSlider(value=0.0, description='mu', max=5.0, min=-5.0), FloatSlider(value=1.0, desc…

interactive(children=(IntSlider(value=1, description='n', max=200, min=1), Dropdown(description='source', opti…

---
## 7. Lognormal($\mu,\sigma^2$): $\ln X$ is Normal

### Meaning
If $Y\sim N(\mu,\sigma^2)$ then $X=e^Y$ is lognormal. Normal effects that **add** in log-space **multiply** in price-space. It is always positive and right-skewed, which is exactly right for prices.

### Derivation of the PDF (change of variables)
$F_X(x)=P(e^Y\le x)=P(Y\le\ln x)=\Phi\big(\frac{\ln x-\mu}\sigma\big)$. Differentiate:
$$f_X(x)=\frac1{x\sigma\sqrt{2\pi}}\exp\Big(-\frac{(\ln x-\mu)^2}{2\sigma^2}\Big),\quad x>0.$$

### Derivation of mean and variance (use the Normal MGF)
$E[e^{kY}]=M_Y(k)=e^{k\mu+k^2\sigma^2/2}$. Therefore
$$E[X]=M_Y(1)=e^{\mu+\sigma^2/2},\qquad E[X^2]=M_Y(2)=e^{2\mu+2\sigma^2}.$$
$$\operatorname{Var}(X)=e^{2\mu+2\sigma^2}-e^{2\mu+\sigma^2}=e^{2\mu+\sigma^2}(e^{\sigma^2}-1).$$

### Median, mode, and the mean-median gap
Median: $P(X\le m)=\tfrac12\Rightarrow\ln m=\mu\Rightarrow m=e^\mu$ (does not depend on $\sigma$). Mode $=e^{\mu-\sigma^2}$.
Mean/median $=e^{\sigma^2/2}$, which grows with $\sigma$: **Mean > Median > Mode** (right skew). Skewness $=(e^{\sigma^2}+2)\sqrt{e^{\sigma^2}-1}$.

### Black-Scholes terminal price (derivation)
Geometric Brownian motion $dS=\mu S\,dt+\sigma S\,dW$. By Ito's lemma on $\ln S$: $d\ln S=(\mu-\tfrac12\sigma^2)dt+\sigma dW$ (the $-\tfrac12\sigma^2$ is the Ito correction). Integrating:
$$\ln S_T=\ln S_0+(\mu-\tfrac12\sigma^2)T+\sigma\sqrt T\,Z,\ Z\sim N(0,1).$$
Under risk-neutral measure $\mu=r$: $S_T=S_0\exp\big((r-\tfrac{\sigma^2}2)T+\sigma\sqrt TZ\big)$, which is lognormal. Check: $E[S_T]=S_0e^{(r-\sigma^2/2)T}e^{\sigma^2T/2}=S_0e^{rT}$ as it should be.

### Where it is used
Stock prices in Black-Scholes, asset sizes, incomes, city sizes, loss severities (body of the distribution).

In [ ]:
def plot_lognormal(mu=0.0, sigma=0.5):
    d = st.lognorm(s=sigma, scale=np.exp(mu)); x = np.linspace(0.001, max(6, d.ppf(.995)), 800)
    fig, ax = plt.subplots(figsize=(11, 4)); ax.plot(x, d.pdf(x), 'k')
    ax.axvline(np.exp(mu - sigma**2), color='g', ls=':', label=f'mode={np.exp(mu-sigma**2):.2f}')
    ax.axvline(np.exp(mu), color='b', ls='--', label=f'median=e^mu={np.exp(mu):.2f}')
    ax.axvline(np.exp(mu + sigma**2/2), color='r', label=f'mean={np.exp(mu+sigma**2/2):.2f}')
    ax.set_title('Lognormal  ' + stats_str(d)); ax.legend(); plt.show()
interact(plot_lognormal, mu=(-1.0, 2.0, 0.05), sigma=(0.05, 1.5, 0.05));

# Black-Scholes terminal price distribution S_T
def plot_bs_terminal(S0=100, r=0.05, sigma=0.20, T=1.0):
    mu_ln = np.log(S0) + (r - sigma**2/2)*T; s_ln = sigma*np.sqrt(T)
    d = st.lognorm(s=s_ln, scale=np.exp(mu_ln)); x = np.linspace(d.ppf(.0005), d.ppf(.9995), 600)
    Z = np.random.default_rng(2).standard_normal(20000); ST = S0*np.exp((r-sigma**2/2)*T + sigma*np.sqrt(T)*Z)
    fig, ax = plt.subplots(figsize=(11, 4)); ax.hist(ST, bins=80, density=True, alpha=.4, label='simulated S_T')
    ax.plot(x, d.pdf(x), 'r', label='lognormal pdf'); ax.axvline(S0*np.exp(r*T), color='k', ls='--', label=f'E[S_T]=S0 e^(rT)={S0*np.exp(r*T):.2f}')
    ax.set_title(f'GBM terminal price: simulated mean={ST.mean():.2f}'); ax.legend(); plt.show()
interact(plot_bs_terminal, S0=(50, 200, 5), r=(0.0, 0.15, 0.005), sigma=(0.05, 0.8, 0.01), T=(0.1, 5.0, 0.1));

interactive(children=(FloatSlider(value=0.0, description='mu', max=2.0, min=-1.0, step=0.05), FloatSlider(valu…

interactive(children=(IntSlider(value=100, description='S0', max=200, min=50, step=5), FloatSlider(value=0.05,…

---
## 8. Exponential($\lambda$): waiting time between random events

### Meaning
If events occur at rate $\lambda$ per unit time (Poisson process), the waiting time until the next event is Exponential($\lambda$).

### Derivation from the Poisson process
Number of events in $[0,t]$ is Poi($\lambda t$). The waiting time $T>t$ exactly when zero events occur in $[0,t]$:
$$P(T>t)=P(N_t=0)=e^{-\lambda t}.$$
So $F(t)=1-e^{-\lambda t}$ and $f(t)=\lambda e^{-\lambda t},\ t\ge0$. Check: $\int_0^\infty\lambda e^{-\lambda t}dt=1$.

### Derivation of mean and variance
Integration by parts ($u=x,\ dv=\lambda e^{-\lambda x}dx$):
$$E[X]=\int_0^\infty x\lambda e^{-\lambda x}dx=\big[-xe^{-\lambda x}\big]_0^\infty+\int_0^\infty e^{-\lambda x}dx=0+\frac1\lambda.$$
$$E[X^2]=\int_0^\infty x^2\lambda e^{-\lambda x}dx=\big[-x^2e^{-\lambda x}\big]_0^\infty+2\int_0^\infty xe^{-\lambda x}dx=\frac2\lambda\cdot E[X]=\frac2{\lambda^2}.$$
$$\operatorname{Var}=\frac2{\lambda^2}-\frac1{\lambda^2}=\frac1{\lambda^2}.$$
Skewness $=2$, excess kurtosis $=6$ (always, regardless of $\lambda$). The median is $\ln2/\lambda<$ mean.

### Memoryless property (proof)
$P(X>s+t\mid X>s)=\dfrac{e^{-\lambda(s+t)}}{e^{-\lambda s}}=e^{-\lambda t}=P(X>t)$. (Continuous version of the Geometric.)

### Hazard rate
$h(t)=\dfrac{f(t)}{1-F(t)}=\dfrac{\lambda e^{-\lambda t}}{e^{-\lambda t}}=\lambda$: constant. This is the only distribution with a constant hazard.

### Where it is used: credit risk with constant hazard rate
Survival probability $S(t)=e^{-\lambda t}$; **probability of default** by time $t$: $PD(t)=1-e^{-\lambda t}$. Credit-spread rule of thumb: $\lambda\approx\text{spread}/(1-R)$. Larger $\lambda$ means earlier default. Also: time between trades, time to next jump.

In [ ]:
def plot_exponential(lam=0.5, t=2.0):
    d = st.expon(scale=1/lam); x = np.linspace(0, 12/lam if lam < 1 else 12, 600)
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
    ax[0].plot(x, d.pdf(x)); ax[0].fill_between(x[x <= t], d.pdf(x[x <= t]), alpha=.3, color='r')
    ax[0].set_title(f'PDF; shaded = PD({t:.1f}) = {d.cdf(t):.3f}')
    ax[1].plot(x, d.sf(x), label='survival e^(-lam t)'); ax[1].plot(x, d.cdf(x), label='PD = 1 - e^(-lam t)')
    ax[1].axvline(t, color='k', ls=':'); ax[1].legend(); ax[1].set_title(stats_str(d))
    ax[2].plot(x, np.full_like(x, lam)); ax[2].set_ylim(0, 3); ax[2].set_title('Hazard rate (constant = lambda)')
    plt.tight_layout(); plt.show()
interact(plot_exponential, lam=(0.05, 3.0, 0.05), t=(0.1, 10.0, 0.1));

interactive(children=(FloatSlider(value=0.5, description='lam', max=3.0, min=0.05, step=0.05), FloatSlider(val…

---
## 9. Gamma($k,\theta$): waiting time until the $k$-th event

### Meaning
The sum of $k$ independent Exponential waiting times (each with mean $\theta$). Also a flexible positive, right-skewed distribution. $k$ = shape, $\theta$ = scale (rate $=1/\theta$).

### Formula and normalisation
$$f(x)=\frac{x^{k-1}e^{-x/\theta}}{\Gamma(k)\theta^k},\ x>0.$$
Normalisation: substitute $t=x/\theta$: $\int_0^\infty x^{k-1}e^{-x/\theta}dx=\theta^k\int_0^\infty t^{k-1}e^{-t}dt=\theta^k\Gamma(k)$. Hence the denominator.

### Derivation of mean and variance
$$E[X^n]=\frac{1}{\Gamma(k)\theta^k}\int x^{k+n-1}e^{-x/\theta}dx=\frac{\theta^{k+n}\Gamma(k+n)}{\Gamma(k)\theta^k}=\theta^n\frac{\Gamma(k+n)}{\Gamma(k)}.$$
Using $\Gamma(z+1)=z\Gamma(z)$:
$E[X]=\theta\,\dfrac{\Gamma(k+1)}{\Gamma(k)}=k\theta$, $\ E[X^2]=\theta^2\dfrac{\Gamma(k+2)}{\Gamma(k)}=k(k+1)\theta^2$.
$$\operatorname{Var}=k(k+1)\theta^2-k^2\theta^2=k\theta^2.$$
Skewness $=2/\sqrt k$, excess kurtosis $=6/k$.

### MGF and the link to Exponential
$M(t)=\dfrac1{\Gamma(k)\theta^k}\int x^{k-1}e^{-x(1/\theta-t)}dx=(1-\theta t)^{-k}$ for $t<1/\theta$.
For $k=1$ this is the Exponential MGF $(1-\theta t)^{-1}$. For independent sum: $(1-\theta t)^{-k_1}(1-\theta t)^{-k_2}=(1-\theta t)^{-(k_1+k_2)}$, so Gamma($k_1,\theta$)+Gamma($k_2,\theta$) = Gamma($k_1+k_2,\theta$). So the sum of $k$ Exponentials is Gamma($k,\theta$). Mean $k\theta$, variance $k\theta^2$ confirm it.

### Role of parameters
$k=1\to$ Exponential. Larger $k\to$ bell-shaped, nearly Normal (skewness $2/\sqrt k\to0$). $\theta$ just stretches the scale.

### Where it is used
Loss severity modelling, Bayesian conjugate prior for a Poisson rate, building block of **Variance-Gamma** and **Student-t** (a Normal with Gamma-distributed precision), CIR-type processes.

In [11]:
def plot_gamma(k=2.0, theta=1.0, normal_approx=False):
    d = st.gamma(k, scale=theta); x = np.linspace(0, d.ppf(.9995), 600)
    fig, ax = plt.subplots(figsize=(11, 4)); ax.plot(x, d.pdf(x), 'k', label='Gamma')
    ax.plot(x, st.expon(scale=theta).pdf(x), 'g:', label='Exponential (k=1)')
    if normal_approx: ax.plot(x, st.norm(k*theta, np.sqrt(k)*theta).pdf(x), 'r--', label='Normal(k theta, k theta^2)')
    ax.set_title(f'Gamma(k={k:.1f}, theta={theta:.1f})  ' + stats_str(d)); ax.legend(); plt.show()
interact(plot_gamma, k=(0.2, 30.0, 0.1), theta=(0.2, 5.0, 0.1));

interactive(children=(FloatSlider(value=2.0, description='k', max=30.0, min=0.2), FloatSlider(value=1.0, descr…

---
## 10. Chi-square $\chi^2(k)$: sum of $k$ squared standard normals

### Meaning
$\chi^2_k=Z_1^2+\dots+Z_k^2$, with $Z_i\sim N(0,1)$ iid. The natural distribution of a sum of squared errors, i.e. of variance estimates.

### Derivation for $k=1$ (change of variables)
$Y=Z^2$. For $y>0$: $P(Y\le y)=P(-\sqrt y\le Z\le\sqrt y)=2\Phi(\sqrt y)-1$. Differentiate:
$$f_Y(y)=2\varphi(\sqrt y)\frac1{2\sqrt y}=\frac{1}{\sqrt{2\pi y}}e^{-y/2}.$$
Compare with Gamma($k_g,\theta$): exponent $y^{k_g-1}=y^{-1/2}\Rightarrow k_g=\tfrac12$, $\theta=2$. So $\chi^2_1=$ Gamma($\tfrac12,2$), and since sums of Gammas with equal scale add shapes:
$$\chi^2_k=\text{Gamma}\Big(\frac k2,2\Big).$$

### Mean and variance (two ways)
From Gamma: mean $=\frac k2\cdot2=k$, variance $=\frac k2\cdot4=2k$.
Directly: $E[Z^2]=1$ so mean $=k$. $E[Z^4]=3$, so $\operatorname{Var}(Z^2)=3-1=2$, and independence gives $\operatorname{Var}=2k$.
Skewness $\sqrt{8/k}$, excess kurtosis $12/k$.

### Where it is used
* **Variance tests:** $(n-1)s^2/\sigma^2\sim\chi^2_{n-1}$ for Normal data.
* **Jarque-Bera normality test:** $JB=\frac n6(S^2+\frac{(K-3)^2}4)\sim\chi^2_2$.
* **Ljung-Box** autocorrelation test, likelihood-ratio tests ($-2\ln\Lambda\sim\chi^2$).
* CIR interest-rate model (non-central $\chi^2$ transition density).

In [12]:
def plot_chi2(k=5, N=20000):
    d = st.chi2(k); x = np.linspace(0, max(15, d.ppf(.999)), 600)
    sim = (np.random.default_rng(3).standard_normal((N, k))**2).sum(axis=1)
    fig, ax = plt.subplots(figsize=(11, 4)); ax.hist(sim, bins=80, density=True, alpha=.4, label='sum of k squared normals (simulated)')
    ax.plot(x, d.pdf(x), 'r', label='chi-square pdf'); ax.set_xlim(0, x[-1]); ax.set_ylim(0, 1.0 if k < 3 else None)
    ax.set_title(f'chi2(k={k})  ' + stats_str(d)); ax.legend(); plt.show()
interact(plot_chi2, k=(1, 40, 1), N=(2000, 100000, 2000));

interactive(children=(IntSlider(value=5, description='k', max=40, min=1), IntSlider(value=20000, description='…

---
## 11. Student's $t(\nu)$: Normal with extra uncertainty about $\sigma$, so fat tails

### Meaning and construction
$$T=\frac{Z}{\sqrt{V/\nu}},\qquad Z\sim N(0,1),\ V\sim\chi^2_\nu\ \text{independent}.$$
Equivalent view: a **scale mixture of normals**. Conditional on a random variance, $T$ is Normal; mixing over many variances creates fat tails. It models "we don't know $\sigma$" (small samples) or "volatility itself fluctuates" (returns).

### Derivation of the PDF (sketch)
Given $V=v$, $T\mid v\sim N(0,\nu/v)$. Integrate out $v$ with the $\chi^2_\nu$ density $\propto v^{\nu/2-1}e^{-v/2}$:
$$f(t)=\int_0^\infty\sqrt{\frac v{2\pi\nu}}e^{-vt^2/(2\nu)}\ \frac{v^{\nu/2-1}e^{-v/2}}{2^{\nu/2}\Gamma(\nu/2)}dv\ \propto\int v^{(\nu+1)/2-1}e^{-v(1+t^2/\nu)/2}dv.$$
This is a Gamma integral, giving $\propto\Gamma\big(\tfrac{\nu+1}2\big)\big(1+t^2/\nu\big)^{-(\nu+1)/2}$. Result:
$$f(t)=\frac{\Gamma(\frac{\nu+1}2)}{\sqrt{\nu\pi}\,\Gamma(\frac\nu2)}\Big(1+\frac{t^2}\nu\Big)^{-\frac{\nu+1}2}.$$
**Tail:** $f(t)\sim|t|^{-(\nu+1)}$, a power law (Normal tail is $e^{-t^2/2}$). That is why moments of order $\ge\nu$ do not exist.

### Derivation of mean and variance
Mean: $E[T]=E[Z]\,E[(V/\nu)^{-1/2}]=0$ (needs $\nu>1$ for $E[V^{-1/2}]$ to be finite).
Variance: by independence $E[T^2]=E[Z^2]\cdot\nu E[1/V]=\nu E[1/V]$. For $V\sim\chi^2_\nu=$Gamma($\nu/2,2$):
$$E[1/V]=\frac{1}{\Gamma(\nu/2)2^{\nu/2}}\int v^{\nu/2-2}e^{-v/2}dv=\frac{\Gamma(\nu/2-1)2^{\nu/2-1}}{\Gamma(\nu/2)2^{\nu/2}}=\frac{1}{2(\nu/2-1)}=\frac1{\nu-2}.$$
(valid for $\nu>2$). Therefore
$$\operatorname{Var}(T)=\frac\nu{\nu-2},\quad\nu>2.$$
Similarly $E[T^4]=3\nu^2E[V^{-2}]=\dfrac{3\nu^2}{(\nu-2)(\nu-4)}$, so kurtosis $=\dfrac{3(\nu-2)}{\nu-4}$ and
$$\text{excess kurtosis}=\frac{3(\nu-2)}{\nu-4}-3=\frac6{\nu-4},\quad\nu>4.$$

### Role of $\nu$ (degrees of freedom)
$\nu=1$: Cauchy. $\nu\to\infty$: Normal. **Daily returns: $\nu\approx3$ to 6.** Rule: only moments of order $<\nu$ exist. With $\nu=3$ the variance is finite but kurtosis is infinite.

### Where it is used
Modelling returns, **t-copulas** (tail dependence), GARCH-t, small-sample confidence intervals and t-tests ($\bar X-\mu)/(s/\sqrt n)\sim t_{n-1}$, fat-tailed VaR.

In [13]:
def plot_t(nu=4.0, show_normal=True, loc=0.0, scale=1.0):
    d = st.t(nu, loc, scale); x = np.linspace(-8, 8, 1000)
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ax[0].plot(x, d.pdf(x), 'k', label=f't(nu={nu:.1f})')
    if show_normal: ax[0].plot(x, st.norm(loc, scale*np.sqrt(nu/(nu-2)) if nu > 2 else scale).pdf(x), 'r--', label='Normal (same variance)' if nu > 2 else 'Normal')
    ax[0].legend(); ax[0].set_title('PDF  ' + stats_str(d))
    xt = np.linspace(2, 12, 400)
    ax[1].semilogy(xt, d.pdf(xt), 'k', label='t tail')
    if show_normal: ax[1].semilogy(xt, st.norm(loc, scale*np.sqrt(nu/(nu-2)) if nu > 2 else scale).pdf(xt), 'r--', label='Normal tail')
    ax[1].set_title('Right tail, log scale (fat tail = much higher)'); ax[1].legend(); ax[1].set_ylim(1e-12, 1)
    plt.tight_layout(); plt.show()
interact(plot_t, nu=(1.0, 40.0, 0.5), loc=(-2.0, 2.0, 0.1), scale=(0.3, 2.0, 0.1));

interactive(children=(FloatSlider(value=4.0, description='nu', max=40.0, min=1.0, step=0.5), Checkbox(value=Tr…

---
## 12. F($d_1,d_2$): ratio of two scaled chi-squares

### Meaning and construction
$$F=\frac{U_1/d_1}{U_2/d_2},\quad U_1\sim\chi^2_{d_1},\ U_2\sim\chi^2_{d_2}\ \text{independent}.$$
It compares two variance estimates. If both come from the same variance, the ratio is near 1.

### Mean (derivation)
By independence $E[F]=\dfrac{E[U_1]}{d_1}\cdot d_2E[1/U_2]=\dfrac{d_1}{d_1}\cdot\dfrac{d_2}{d_2-2}=\dfrac{d_2}{d_2-2}$ for $d_2>2$ (using $E[1/V]=1/(\nu-2)$ from the t-section). Note it depends only on $d_2$.

### Variance
$\operatorname{Var}(F)=\dfrac{2d_2^2(d_1+d_2-2)}{d_1(d_2-2)^2(d_2-4)}$ for $d_2>4$ (from $E[U_2^{-2}]=1/((d_2-2)(d_2-4))$).

### Relationships
$t_\nu^2=F(1,\nu)$. As $d_2\to\infty$, $d_1F\to\chi^2_{d_1}$.

### Where it is used
Regression **joint-significance tests** ($F=\frac{(R^2/k)}{(1-R^2)/(n-k-1)}$), ANOVA, comparing two variances (e.g. volatility before/after an event), Chow structural-break tests.

In [14]:
def plot_F(d1=5, d2=20):
    d = st.f(d1, d2); x = np.linspace(0.001, 6, 800)
    fig, ax = plt.subplots(figsize=(11, 4)); ax.plot(x, d.pdf(x))
    c = d.ppf(.95); ax.fill_between(x[x >= c], d.pdf(x[x >= c]), color='r', alpha=.3, label=f'5% critical value = {c:.2f}')
    ax.set_title(f'F({d1},{d2})  ' + stats_str(d)); ax.legend(); plt.show()
interact(plot_F, d1=(1, 50, 1), d2=(1, 100, 1));

interactive(children=(IntSlider(value=5, description='d1', max=50, min=1), IntSlider(value=20, description='d2…

---
## 13. Beta($\alpha,\beta$): a distribution on $[0,1]$

### Meaning
A flexible distribution for **probabilities and proportions**. The shape can be U-shaped, bell-shaped, skewed or flat depending on $\alpha,\beta$. Beta is the conjugate prior of the Bernoulli/Binomial $p$.

### Formula and normalisation
$$f(x)=\frac{x^{\alpha-1}(1-x)^{\beta-1}}{B(\alpha,\beta)},\ 0<x<1,\qquad B(\alpha,\beta)=\int_0^1t^{\alpha-1}(1-t)^{\beta-1}dt=\frac{\Gamma(\alpha)\Gamma(\beta)}{\Gamma(\alpha+\beta)}.$$

### Derivation of mean and variance
$$E[X^n]=\frac{1}{B(\alpha,\beta)}\int x^{\alpha+n-1}(1-x)^{\beta-1}dx=\frac{B(\alpha+n,\beta)}{B(\alpha,\beta)}=\frac{\Gamma(\alpha+n)\Gamma(\alpha+\beta)}{\Gamma(\alpha)\Gamma(\alpha+\beta+n)}.$$
$n=1$: $E[X]=\dfrac{\alpha}{\alpha+\beta}$.
$n=2$: $E[X^2]=\dfrac{\alpha(\alpha+1)}{(\alpha+\beta)(\alpha+\beta+1)}$.
$$\operatorname{Var}=\frac{\alpha(\alpha+1)}{(\alpha+\beta)(\alpha+\beta+1)}-\frac{\alpha^2}{(\alpha+\beta)^2}=\frac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)}.$$
With $s=\alpha+\beta$ ("sample size") and $m=\alpha/s$: $\operatorname{Var}=m(1-m)/(s+1)$, so **larger $\alpha+\beta$ means tighter**.

### Shapes
$\alpha=\beta=1$: Uniform. $\alpha,\beta<1$: U-shaped (mass at 0 and 1). $\alpha,\beta>1$: bell. $\alpha<\beta$: skewed towards 0.

### Link to Gamma and Bayesian updating
If $G_1\sim$Gamma($\alpha,1$), $G_2\sim$Gamma($\beta,1$) independent, then $G_1/(G_1+G_2)\sim$Beta($\alpha,\beta$).
**Bayes:** prior Beta($\alpha,\beta$) on $p$; observe $k$ successes in $n$ trials. Posterior $\propto p^{\alpha-1}(1-p)^{\beta-1}p^k(1-p)^{n-k}$ = Beta($\alpha+k,\beta+n-k$). Neat: just add counts.

### Where it is used
**Recovery rates** (loss given default), uncertain probabilities (PD uncertainty), prior for Bernoulli/Binomial $p$, proportions, Dirichlet generalisation for portfolio weights.

In [15]:
def plot_beta(alpha=2.0, beta=5.0, k_succ=0, n_trials=0):
    prior = st.beta(alpha, beta); post = st.beta(alpha+k_succ, beta+n_trials-k_succ)
    x = np.linspace(0.001, 0.999, 800); fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(x, prior.pdf(x), 'b', label='prior Beta(a,b)')
    if n_trials > 0 and k_succ <= n_trials: ax.plot(x, post.pdf(x), 'r', label=f'posterior Beta({alpha+k_succ:.1f},{beta+n_trials-k_succ:.1f})')
    ax.set_ylim(0, min(12, 1.2*max(prior.pdf(x).max(), post.pdf(x).max()) if n_trials else 1.2*prior.pdf(x).max()))
    ax.set_title('Beta  ' + stats_str(prior) + '   (Bayesian updating: set trials/successes)'); ax.legend(); plt.show()
interact(plot_beta, alpha=(0.1, 20.0, 0.1), beta=(0.1, 20.0, 0.1), k_succ=(0, 100, 1), n_trials=(0, 100, 1));

interactive(children=(FloatSlider(value=2.0, description='alpha', max=20.0, min=0.1), FloatSlider(value=5.0, d…

---
## 14. Weibull($k,\lambda$): time-to-failure with a changing hazard rate

### Meaning
Generalises the Exponential by allowing the failure rate to change with age. $k$ = shape, $\lambda$ = scale (characteristic life).

### Derivation from the hazard rate (this is the cleanest route)
Define the hazard as a power law $h(t)=\dfrac k\lambda\Big(\dfrac t\lambda\Big)^{k-1}\propto t^{k-1}$. Survival and hazard are linked by $S(t)=\exp\!\big(-\int_0^th(s)ds\big)$ (because $h=-S'/S$). Integrate:
$$\int_0^th(s)ds=\Big(\frac t\lambda\Big)^k\ \Rightarrow\ S(t)=e^{-(t/\lambda)^k},\quad f(t)=\frac k\lambda\Big(\frac t\lambda\Big)^{k-1}e^{-(t/\lambda)^k}.$$
* $k<1$: hazard **falls** with time ("infant mortality"; survivors get safer).
* $k=1$: constant hazard, **Exponential** with mean $\lambda$.
* $k>1$: hazard **rises** ("wear-out"; ageing).

### Derivation of the mean (substitution)
$E[X]=\int_0^\infty S(t)dt=\int_0^\infty e^{-(t/\lambda)^k}dt$. Let $u=(t/\lambda)^k$, $t=\lambda u^{1/k}$, $dt=\frac\lambda ku^{1/k-1}du$:
$$E[X]=\frac\lambda k\int_0^\infty u^{1/k-1}e^{-u}du=\frac\lambda k\Gamma\Big(\frac1k\Big)=\lambda\Gamma\Big(1+\frac1k\Big).$$
(using $\Gamma(1+z)=z\Gamma(z)$). In general $E[X^n]=\lambda^n\Gamma(1+n/k)$, so
$$\operatorname{Var}=\lambda^2\Big[\Gamma\Big(1+\frac2k\Big)-\Gamma\Big(1+\frac1k\Big)^2\Big].$$

### Where it is used
Survival and credit models where default risk **depends on age** (seasoning of loans, bond issuers), reliability engineering (component failure; your EE background: bathtub curve = $k<1$ then $k=1$ then $k>1$), wind-speed modelling, extreme-value Type III.

In [16]:
def plot_weibull(k=1.5, lam=1.0):
    d = st.weibull_min(k, scale=lam); x = np.linspace(0.001, 4*lam, 600)
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
    ax[0].plot(x, d.pdf(x)); ax[0].set_title('PDF  ' + stats_str(d)); ax[0].set_ylim(0, 3)
    ax[1].plot(x, d.sf(x)); ax[1].set_title('Survival S(t) = exp(-(t/lam)^k)')
    h = (k/lam)*(x/lam)**(k-1); ax[2].plot(x, h); ax[2].set_ylim(0, 6)
    ax[2].set_title('Hazard ~ ' + ('falling' if k < 1 else 'constant' if k == 1 else 'rising') + f' (k={k:.2f})')
    plt.tight_layout(); plt.show()
interact(plot_weibull, k=(0.3, 6.0, 0.05), lam=(0.3, 3.0, 0.1));

interactive(children=(FloatSlider(value=1.5, description='k', max=6.0, min=0.3, step=0.05), FloatSlider(value=…

---
## 15. Laplace($\mu,b$): two exponentials back to back

### Meaning
The "double exponential": exponential decay on both sides of a sharp peak at $\mu$. Sharper peak and heavier tails than the Normal.

### Formula and normalisation
$$f(x)=\frac1{2b}e^{-|x-\mu|/b}.$$
$\int f=2\cdot\frac1{2b}\int_0^\infty e^{-y/b}dy=2\cdot\frac1{2b}\cdot b=1$. ✓.

### Mean and variance
Symmetric about $\mu$, so mean $=\mu$. With $y=x-\mu$:
$$E[y^2]=2\cdot\frac1{2b}\int_0^\infty y^2e^{-y/b}dy=\frac1b\cdot\Gamma(3)b^3=2b^2.$$
(Gamma integral: $\int_0^\infty y^2e^{-y/b}dy=2b^3$.) So $\operatorname{Var}=2b^2$.
Fourth moment: $E[y^4]=\frac1b\Gamma(5)b^5=24b^4$, kurtosis $=24b^4/(2b^2)^2=6$, **excess kurtosis $=3$**.

### Link to L1 regression
Log-likelihood of Laplace data: $\ell=-n\ln(2b)-\frac1b\sum|x_i-\mu|$. Maximising over $\mu$ = **minimising absolute error** $\sum|x_i-\mu|$, which is solved by the **median**. (Normal likelihood gives squared error and the mean.) So Laplace errors $\leftrightarrow$ L1/median/quantile regression, and Laplace prior $\leftrightarrow$ LASSO.

### Where it is used
Modelling returns with a peaked centre, robust regression, LASSO priors, differential privacy noise, Value-at-Risk with exponential tails.

In [17]:
def plot_laplace(mu=0.0, b=1.0):
    d = st.laplace(mu, b); x = np.linspace(-10, 10, 1000); nm = st.norm(mu, np.sqrt(2)*b)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(x, d.pdf(x), 'k', label='Laplace'); ax[0].plot(x, nm.pdf(x), 'r--', label='Normal, same variance'); ax[0].legend()
    ax[0].set_title('PDF  ' + stats_str(d))
    ax[1].semilogy(x, d.pdf(x), 'k'); ax[1].semilogy(x, nm.pdf(x), 'r--'); ax[1].set_ylim(1e-8, 1); ax[1].set_title('Log scale: heavier tails')
    plt.tight_layout(); plt.show()
interact(plot_laplace, mu=(-3.0, 3.0, 0.1), b=(0.2, 3.0, 0.1));

interactive(children=(FloatSlider(value=0.0, description='mu', max=3.0, min=-3.0), FloatSlider(value=1.0, desc…

---
# PART C: THE FAT-TAIL FAMILY (the important ones for risk)

**Why they matter:** market crashes (-20% days) are impossible under a Normal (probability $\sim10^{-88}$) yet happen every few decades. Fat-tailed distributions assign realistic probability to them.

---
## 16. Cauchy($x_0,\gamma$)

### Meaning
The "worst-behaved" bell-shaped distribution: tails so fat that the mean does not exist. In physics it is the **Lorentzian** resonance line shape. $x_0$ = median/mode, $\gamma$ = half-width at half-maximum.

### Formula and normalisation
$$f(x)=\frac1{\pi\gamma\big[1+\big(\frac{x-x_0}\gamma\big)^2\big]},\qquad F(x)=\frac12+\frac1\pi\arctan\Big(\frac{x-x_0}\gamma\Big).$$
Check: $\int\frac{dx}{1+x^2}=[\arctan x]_{-\infty}^\infty=\pi$, so the constant is $1/\pi$ (for $\gamma=1,x_0=0$).

### Why the mean is undefined
$E|X|=\frac2\pi\int_0^\infty\frac{x}{1+x^2}dx=\frac1\pi\big[\ln(1+x^2)\big]_0^\infty=\infty$. The integral diverges logarithmically. By symmetry one might say "mean = 0" but the answer depends on how you take the limit, so it is **undefined**. Variance is infinite ($\int x^2/(1+x^2)$ does not converge).

### Derivation: ratio of two independent standard normals
$(Z_1,Z_2)$ is rotationally symmetric, so its angle $\Theta$ is uniform. $Z_1/Z_2=\tan\Theta$ (or $\cot$). If $\Theta\sim U(-\pi/2,\pi/2)$ then $P(\tan\Theta\le x)=\frac{\arctan x+\pi/2}{\pi}$, exactly the Cauchy CDF. Equivalent to the Student-$t$ with $\nu=1$ (since $T=Z/\sqrt{V/1}=Z/|Z'|$).

### The shocking property: averaging does not help
Characteristic function: $\varphi(t)=e^{ix_0t-\gamma|t|}$. For the sample mean $\bar X=\frac1n\sum X_i$: $\varphi_{\bar X}(t)=\varphi(t/n)^n=e^{ix_0t-\gamma|t|}$, **the same Cauchy**. The average of 1,000,000 Cauchy draws is as noisy as one draw. The Law of Large Numbers and CLT both fail.

### Where it is used
Mostly a stress-test and teaching example ("what if tails are as bad as possible"); physics line shapes; ratio of Normals; robust-statistics benchmarks; as a heavy-tailed prior.

In [18]:
def plot_cauchy(x0=0.0, gamma=1.0, n=2000, seed=1):
    d = st.cauchy(x0, gamma); x = np.linspace(-10, 10, 1000)
    rng = np.random.default_rng(seed)
    c = d.rvs(n, random_state=rng); g = st.norm(x0, gamma).rvs(n, random_state=rng)
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ax[0].plot(x, d.pdf(x), 'k', label='Cauchy'); ax[0].plot(x, st.norm(x0, gamma).pdf(x), 'r--', label='Normal (same loc/scale)')
    ax[0].legend(); ax[0].set_title('PDF: fatter tails')
    idx = np.arange(1, n+1); ax[1].plot(idx, np.cumsum(c)/idx, 'k', label='Cauchy running mean'); ax[1].plot(idx, np.cumsum(g)/idx, 'r', label='Normal running mean')
    ax[1].set_ylim(x0-5*gamma, x0+5*gamma); ax[1].legend(); ax[1].set_title('Running mean: Normal converges, Cauchy never does (change seed!)')
    plt.tight_layout(); plt.show()
interact(plot_cauchy, x0=(-3.0, 3.0, 0.1), gamma=(0.2, 3.0, 0.1), n=(100, 20000, 100), seed=(0, 50, 1));

interactive(children=(FloatSlider(value=0.0, description='x0', max=3.0, min=-3.0), FloatSlider(value=1.0, desc…

---
## 17. Pareto($x_m,\alpha$): the power law

### Meaning
"The rich get richer": the probability of exceeding $x$ decays as a power of $x$, not exponentially. $x_m$ = minimum value, $\alpha$ = **tail index** (smaller $\alpha$ = fatter tail).

### Formula
$$P(X>x)=\Big(\frac{x_m}x\Big)^\alpha,\ x\ge x_m,\qquad f(x)=\frac{\alpha x_m^\alpha}{x^{\alpha+1}}.$$
On a log-log plot the survival function is a **straight line with slope $-\alpha$**. (The Normal curves downward.)

### Derivation of moments (the key rule: the $n$-th moment exists only if $n<\alpha$)
$$E[X^n]=\int_{x_m}^\infty x^n\frac{\alpha x_m^\alpha}{x^{\alpha+1}}dx=\alpha x_m^\alpha\int_{x_m}^\infty x^{n-\alpha-1}dx=\alpha x_m^\alpha\Big[\frac{x^{n-\alpha}}{n-\alpha}\Big]_{x_m}^\infty.$$
This converges only if $n-\alpha<0$, giving $E[X^n]=\dfrac{\alpha x_m^n}{\alpha-n}$.
* **Mean** ($n=1$, needs $\alpha>1$): $\mu=\dfrac{\alpha x_m}{\alpha-1}$.
* $n=2$ (needs $\alpha>2$): $E[X^2]=\dfrac{\alpha x_m^2}{\alpha-2}$.
$$\operatorname{Var}=\frac{\alpha x_m^2}{\alpha-2}-\frac{\alpha^2x_m^2}{(\alpha-1)^2}=\alpha x_m^2\,\frac{(\alpha-1)^2-\alpha(\alpha-2)}{(\alpha-1)^2(\alpha-2)}=\frac{\alpha x_m^2}{(\alpha-1)^2(\alpha-2)}.$$

### Famous values of $\alpha$
* $\alpha\approx1.16$: the **80/20 rule** (top 20% hold 80%).
* $\alpha\approx3$: stock-return tails ("**inverse cubic law**"). Variance finite, but 4th moment (kurtosis) infinite.
* $\alpha\le2$: infinite variance. $\alpha\le1$: infinite mean.

### Estimating $\alpha$: Hill estimator
Maximum likelihood on the largest $k$ observations: $\hat\alpha=\Big[\frac1k\sum_{i=1}^k\ln\frac{X_{(i)}}{X_{(k+1)}}\Big]^{-1}$ (derive by maximising $\sum\ln f$ above threshold; it equals $n/\sum\ln(x_i/x_m)$).

### Where it is used
Operational-risk losses, insurance mega-claims, wealth/income tails, trading volume, city sizes, return tails.

In [19]:
def plot_pareto(xm=1.0, alpha=3.0):
    d = st.pareto(alpha, scale=xm); x = np.logspace(np.log10(xm), np.log10(xm)+3, 500)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    xl = np.linspace(xm, xm*6, 400); ax[0].plot(xl, d.pdf(xl), 'k'); ax[0].set_title('PDF')
    ax[1].loglog(x, d.sf(x), 'k', label=f'Pareto: slope = -alpha = {-alpha:.2f}')
    ax[1].loglog(x, st.norm(0, 1).sf(x/xm), 'r--', label='Normal tail (curves down)'); ax[1].set_ylim(1e-12, 2); ax[1].legend(); ax[1].set_title('Survival, log-log')
    n = np.arange(1, 7); ok = n < alpha
    ax[2].bar(n[ok], np.ones(ok.sum()), color='g', label='moment exists'); ax[2].bar(n[~ok], np.ones((~ok).sum()), color='r', label='infinite')
    ax[2].set_xlabel('moment order n'); ax[2].set_yticks([]); ax[2].legend(); ax[2].set_title('Moments exist only for n < alpha')
    plt.suptitle(stats_str(d)); plt.tight_layout(); plt.show()
interact(plot_pareto, xm=(0.5, 5.0, 0.1), alpha=(0.6, 6.0, 0.05));

interactive(children=(FloatSlider(value=1.0, description='xm', max=5.0, min=0.5), FloatSlider(value=3.0, descr…

---
## 18. Levy($\mu,c$)

### Meaning
The distribution of the **first time a Brownian motion hits a level** $a$ (with $c=a^2$). One-sided with a very heavy tail.

### Formula
$$f(x)=\sqrt{\frac c{2\pi}}\frac{e^{-\frac c{2(x-\mu)}}}{(x-\mu)^{3/2}},\ x>\mu.$$

### Derivation from Brownian first passage (reflection principle)
Let $T_a=\inf\{t:B_t=a\}$, $a>0$. Reflecting the path after $T_a$ shows $P(T_a\le t)=2P(B_t\ge a)$. Since $B_t\sim N(0,t)$:
$$F(t)=2\Big[1-\Phi\Big(\frac a{\sqrt t}\Big)\Big].$$
Differentiate (chain rule, $\varphi$ = standard normal pdf): $f(t)=2\varphi(a/\sqrt t)\cdot\frac{a}{2t^{3/2}}=\frac{a}{\sqrt{2\pi}\,t^{3/2}}e^{-a^2/(2t)}$, which is Levy with $c=a^2$.
**Shortcut for sampling:** if $Z\sim N(0,1)$ then $c/Z^2\sim$ Levy($0,c$). (Check: $P(c/Z^2\le x)=P(|Z|\ge\sqrt{c/x})=2[1-\Phi(\sqrt{c/x})]$. ✓)

### Moments
For large $x$, $f(x)\sim x^{-3/2}$ so $\int x f(x)dx\sim\int x^{-1/2}dx=\infty$: **mean and variance infinite**. Tail $P(X>x)\sim\sqrt{2c/(\pi x)}\propto x^{-1/2}$: an $\alpha=\tfrac12$ power law.

### Where it is used
Hitting-time modelling (barrier options, first-passage default in the Black-Cox structural credit model), $\alpha=\tfrac12$ stable law, physics of diffusion.

In [20]:
def plot_levy(c=1.0, mu=0.0, N=20000):
    d = st.levy(loc=mu, scale=c); x = np.linspace(mu+0.01, mu+10*c+5, 800)
    sim = mu + c/np.random.default_rng(4).standard_normal(N)**2
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.hist(sim[sim < x[-1]], bins=120, density=True, alpha=.4, label='simulated c/Z^2'); ax.plot(x, d.pdf(x), 'r', label='Levy pdf')
    ax.set_xlim(mu, x[-1]); ax.legend(); ax.set_title(f'Levy(mu={mu:.1f}, c={c:.1f}): mean and variance infinite; median={d.median():.2f}')
    plt.show()
interact(plot_levy, c=(0.1, 5.0, 0.1), mu=(-2.0, 2.0, 0.1), N=(2000, 100000, 2000));

interactive(children=(FloatSlider(value=1.0, description='c', max=5.0, min=0.1), FloatSlider(value=0.0, descri…

---
## 19. $\alpha$-stable family $S(\alpha,\beta,\gamma,\delta)$: the unifying idea

### Meaning
"Stable" means: **the sum of iid variables from the family stays in the family** (only location and scale change). These are the only possible limits of normalised sums of iid variables (generalised CLT). If the variance is finite you get the Normal; if tails are power-law with index $\alpha<2$ you get a stable law with that $\alpha$.

### Definition via characteristic function
$$\varphi(t)=\exp\Big(i\delta t-\gamma^\alpha|t|^\alpha\Big[1-i\beta\,\mathrm{sgn}(t)\tan\frac{\pi\alpha}2\Big]\Big)\quad(\alpha\ne1).$$
* $\alpha\in(0,2]$: **tail index** (tail $\sim|x|^{-\alpha}$); smaller $\alpha$ = fatter.
* $\beta\in[-1,1]$: skewness. $\gamma>0$: scale. $\delta$: location.

### Why it is stable (derivation)
Take $n$ iid copies (take $\beta=0,\delta=0$ for simplicity). Then $\varphi_{\sum}(t)=\varphi(t)^n=\exp(-n\gamma^\alpha|t|^\alpha)=\exp(-(n^{1/\alpha}\gamma)^\alpha|t|^\alpha)$. This is the same family with scale $n^{1/\alpha}\gamma$. So $\sum X_i\overset d=n^{1/\alpha}X$. Compare: for the Normal ($\alpha=2$) sums scale as $\sqrt n$; for Cauchy ($\alpha=1$) as $n$; for fat tails $\alpha<2$ sums grow **faster** than $\sqrt n$, dominated by the single largest shock.

### Special cases (verify from $\varphi$)
* $\alpha=2$: $\varphi=e^{-\gamma^2t^2}$, which is Normal with variance $2\gamma^2$ (tan term vanishes).
* $\alpha=1,\beta=0$: $\varphi=e^{-\gamma|t|}$, **Cauchy**.
* $\alpha=\tfrac12,\beta=1$: **Levy**.

### Moments
For $\alpha<2$, variance is infinite; for $\alpha\le1$, the mean is also undefined; $E|X|^p<\infty$ only for $p<\alpha$.

### History and use
Mandelbrot (1963) proposed it for cotton prices, the birth of the fat-tail view of markets. Downsides: infinite variance clashes with the empirical $\alpha\approx3$ of equities; no closed-form pdf (computed by numerically inverting $\varphi$). Used for stress testing, heavy-tailed noise models, signal processing.

In [21]:
# NOTE: stable pdfs are computed numerically (slower), so the sliders update a bit slower.
from scipy.stats import levy_stable
def plot_stable(alpha=1.7, beta=0.0, gamma=1.0, delta=0.0):
    alpha = min(max(alpha, 0.6), 2.0)
    x = np.linspace(-10, 10, 250)
    pdf = levy_stable.pdf(x, alpha, beta, loc=delta, scale=gamma)
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ax[0].plot(x, pdf, 'k', label=f'stable alpha={alpha:.2f}'); ax[0].plot(x, st.norm(delta, np.sqrt(2)*gamma).pdf(x), 'r--', label='Normal (alpha=2, same gamma)')
    ax[0].plot(x, st.cauchy(delta, gamma).pdf(x), 'g:', label='Cauchy (alpha=1)'); ax[0].legend(); ax[0].set_title('PDF'); ax[0].set_ylim(0, 0.6)
    ax[1].semilogy(x, pdf, 'k'); ax[1].semilogy(x, st.norm(delta, np.sqrt(2)*gamma).pdf(x), 'r--'); ax[1].set_ylim(1e-7, 1); ax[1].set_title('Log scale: power-law tail unless alpha=2')
    plt.tight_layout(); plt.show()
interact(plot_stable, alpha=(0.6, 2.0, 0.05), beta=(-1.0, 1.0, 0.1), gamma=(0.3, 3.0, 0.1), delta=(-3.0, 3.0, 0.1));

interactive(children=(FloatSlider(value=1.7, description='alpha', max=2.0, min=0.6, step=0.05), FloatSlider(va…

---
# PART D: EXTREME-VALUE DISTRIBUTIONS

The Normal CLT describes **averages**. Extreme-value theory (EVT) is the analogous theory for **maxima** and **exceedances**, which is exactly what tail risk needs.

---
## 20. GEV (Generalised Extreme Value): distribution of block maxima

### Meaning
Take a big block of data (e.g. 250 daily losses = 1 year) and record the **maximum**. Repeat for many blocks. The block maxima follow (approximately) a GEV, regardless of the parent distribution (**Fisher-Tippett-Gnedenko theorem**, the "CLT for maxima").

### Formula
$$H_\xi(x)=\exp\Big(-(1+\xi x)^{-1/\xi}\Big),\ 1+\xi x>0,\qquad\text{with location }\mu\text{ and scale }\sigma:\ x\to\frac{x-\mu}\sigma.$$
$\xi=0$ is read as the limit $\exp(-e^{-x})$.

### Derivation idea
For iid $X_i$ with CDF $F$, $P(\max_{i\le n}X_i\le x)=F(x)^n$. We look for constants $a_n>0,b_n$ with $F(a_nx+b_n)^n\to H(x)$.
**Example (Exponential parent):** $F(x)=1-e^{-x}$. Choose $b_n=\ln n$, $a_n=1$:
$$F(x+\ln n)^n=\Big(1-\frac{e^{-x}}n\Big)^n\to\exp(-e^{-x}),$$
the **Gumbel** distribution ($\xi=0$). A Pareto parent gives Fréchet, a bounded parent gives Weibull-type.

### The three types (tail index $\xi$)
* $\xi>0$: **Fréchet**, fat tails, $\xi=1/\alpha$ (Pareto, Student-t, Cauchy parents). *The relevant one for financial losses.* Student-t with $\nu=3$ gives $\xi\approx1/3$.
* $\xi=0$: **Gumbel**, exponential-like tails (Normal, Exponential, Lognormal parents).
* $\xi<0$: **Weibull type**, bounded upper tail (e.g. Uniform).
Moments: mean finite if $\xi<1$, variance finite if $\xi<\tfrac12$.

### Where it is used
Annual worst-day losses, stress scenarios, return-period calculations ("1-in-100 year loss"), catastrophe insurance, flood / wind engineering.

*Note: SciPy uses the shape $c=-\xi$.*

In [22]:
def plot_gev(xi=0.3, mu=0.0, sigma=1.0):
    d = st.genextreme(-xi, loc=mu, scale=sigma); x = np.linspace(-4, 12, 800)
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ax[0].plot(x, d.pdf(x), 'k', label=f'GEV xi={xi:.2f} (' + ('Frechet' if xi > 0.01 else 'Gumbel' if abs(xi) <= 0.01 else 'Weibull-type') + ')')
    ax[0].plot(x, st.gumbel_r(mu, sigma).pdf(x), 'g:', label='Gumbel (xi=0)'); ax[0].legend(); ax[0].set_title('PDF  ' + stats_str(d))
    xs = np.linspace(0, 40, 400); ax[1].semilogy(xs, d.sf(xs), 'k'); ax[1].semilogy(xs, st.gumbel_r(mu, sigma).sf(xs), 'g:'); ax[1].set_ylim(1e-8, 1); ax[1].set_title('Tail P(max > x): xi>0 is far fatter')
    plt.tight_layout(); plt.show()
interact(plot_gev, xi=(-0.5, 1.0, 0.02), mu=(-2.0, 3.0, 0.1), sigma=(0.3, 3.0, 0.1));

# Block-maxima experiment: simulate "daily losses", take the max of each block, fit a GEV
def gev_blocks(parent='student-t(3)', block=250, n_blocks=300, seed=5):
    rng = np.random.default_rng(seed); n = block*n_blocks
    data = {'normal': rng.standard_normal(n), 'student-t(3)': rng.standard_t(3, n), 'laplace': rng.laplace(size=n)}[parent].reshape(n_blocks, block)
    mx = data.max(axis=1); c, loc, sc = st.genextreme.fit(mx); xi = -c
    fig, ax = plt.subplots(figsize=(11, 4)); ax.hist(mx, bins=40, density=True, alpha=.4, label='block maxima')
    x = np.linspace(mx.min(), mx.max(), 400); ax.plot(x, st.genextreme(c, loc, sc).pdf(x), 'r', label=f'fitted GEV: xi={xi:.2f}')
    ax.set_title(f'{parent}: expected xi = ' + {'normal': '0 (Gumbel)', 'student-t(3)': '1/3', 'laplace': '0 (Gumbel)'}[parent]); ax.legend(); plt.show()
interact(gev_blocks, parent=['normal', 'student-t(3)', 'laplace'], block=(20, 500, 10), n_blocks=(50, 1000, 50), seed=(0, 30, 1));

interactive(children=(FloatSlider(value=0.3, description='xi', max=1.0, min=-0.5, step=0.02), FloatSlider(valu…

interactive(children=(Dropdown(description='parent', index=1, options=('normal', 'student-t(3)', 'laplace'), v…

---
## 21. GPD (Generalised Pareto): excesses over a high threshold

### Meaning
Instead of one maximum per block (wasteful), use **all** observations above a high threshold $u$. By the **Pickands-Balkema-de Haan theorem**, the excesses $Y=X-u$ (given $X>u$) are approximately GPD for any reasonable parent distribution. This is "peaks over threshold" (POT), the industry standard for tail risk.

### Formula
$$F(y)=1-\Big(1+\frac{\xi y}\beta\Big)^{-1/\xi},\ y>0,\qquad f(y)=\frac1\beta\Big(1+\frac{\xi y}\beta\Big)^{-1/\xi-1}.$$
$\xi$ = shape (tail index, same $\xi$ as GEV), $\beta$ = scale. $\xi\to0$ gives the Exponential $1-e^{-y/\beta}$. $\xi>0$: Pareto-type; $\xi<0$: bounded support.

### Derivation of the mean
Using $E[Y]=\int_0^\infty P(Y>y)dy$ and $P(Y>y)=(1+\xi y/\beta)^{-1/\xi}$:
$$E[Y]=\int_0^\infty\Big(1+\frac{\xi y}\beta\Big)^{-1/\xi}dy=\frac\beta\xi\Big[\frac{(1+\xi y/\beta)^{1-1/\xi}}{1-1/\xi}\Big]_0^\infty=\frac\beta\xi\cdot\frac{\xi}{\xi-1}\,\big[0-1\big]\ \Rightarrow\ \boxed{E[Y]=\frac\beta{1-\xi}}\quad(\xi<1).$$
(The upper limit vanishes when $1-1/\xi<0$, i.e. $0<\xi<1$.)
Similarly $E[Y^2]=\dfrac{2\beta^2}{(1-\xi)(1-2\xi)}$, so
$$\operatorname{Var}(Y)=\frac{\beta^2}{(1-\xi)^2(1-2\xi)},\quad\xi<\tfrac12.$$

### Derivation: VaR and Expected Shortfall from POT
Let $N$ = total observations, $N_u$ = number above $u$. Then for $x>u$:
$P(X>x)=P(X>u)\,P(Y>x-u\mid X>u)\approx\dfrac{N_u}N\Big(1+\xi\dfrac{x-u}\beta\Big)^{-1/\xi}$.
Set this equal to $1-q$ and solve for $x=\mathrm{VaR}_q$:
$$\boxed{\mathrm{VaR}_q=u+\frac\beta\xi\Big[\Big(\frac N{N_u}(1-q)\Big)^{-\xi}-1\Big]}$$
Expected shortfall (mean loss beyond VaR). The excess over VaR is again GPD with scale $\beta+\xi(\mathrm{VaR}_q-u)$, so by the mean formula:
$$\boxed{\mathrm{ES}_q=\mathrm{VaR}_q+\frac{\beta+\xi(\mathrm{VaR}_q-u)}{1-\xi}=\frac{\mathrm{VaR}_q+\beta-\xi u}{1-\xi}}$$
Note $\mathrm{ES}/\mathrm{VaR}\to1/(1-\xi)$ for far tails: with $\xi=0.33$ ES is about 1.5x VaR; for a Normal it is only about 1.15x.

### Where it is used
99.9% VaR and ES estimation, operational-risk severity tails, insurance excess-of-loss pricing, regulatory capital (Basel, Solvency II).

In [23]:
def plot_gpd(xi=0.3, beta=1.0):
    d = st.genpareto(xi, scale=beta); y = np.linspace(0, 15, 600)
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ax[0].plot(y, d.pdf(y), 'k', label=f'GPD xi={xi:.2f}'); ax[0].plot(y, st.expon(scale=beta).pdf(y), 'g:', label='Exponential (xi=0)')
    ax[0].legend(); ax[0].set_ylim(0, 1.2/beta); ax[0].set_title('PDF  ' + stats_str(d))
    ax[1].semilogy(y, d.sf(y), 'k'); ax[1].semilogy(y, st.expon(scale=beta).sf(y), 'g:'); ax[1].set_ylim(1e-8, 1); ax[1].set_title('Excess survival P(Y>y), log scale')
    plt.tight_layout(); plt.show()
interact(plot_gpd, xi=(-0.5, 1.0, 0.02), beta=(0.3, 3.0, 0.1));

# Peaks-over-threshold demo: fit GPD to simulated fat-tailed losses and estimate VaR / ES
def pot_demo(nu=3.0, N=5000, u_quantile=0.95, q=0.999, seed=7):
    rng = np.random.default_rng(seed); L = rng.standard_t(nu, N)           # losses (right tail)
    u = np.quantile(L, u_quantile); exc = L[L > u] - u; Nu = len(exc)
    xi, _, beta = st.genpareto.fit(exc, floc=0)
    var = u + beta/xi*(((N/Nu)*(1-q))**(-xi) - 1) if abs(xi) > 1e-6 else u - beta*np.log((N/Nu)*(1-q))
    es = (var + beta - xi*u)/(1-xi) if xi < 1 else np.inf
    t = st.t(nu); true_var = t.ppf(q); true_es = (t.pdf(true_var)*(nu + true_var**2)/(nu-1))/(1-q) if nu > 1 else np.inf
    nvar = st.norm.ppf(q); nes = st.norm.pdf(nvar)/(1-q)
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    y = np.linspace(0, exc.max(), 300); ax[0].hist(exc, bins=30, density=True, alpha=.4, label=f'{Nu} excesses over u={u:.2f}')
    ax[0].plot(y, st.genpareto(xi, scale=beta).pdf(y), 'r', label=f'fitted GPD xi={xi:.2f} (true ~{1/nu:.2f}), beta={beta:.2f}'); ax[0].legend()
    labels = ['Normal', 'POT-GPD', 'True t']; ax[1].bar(np.arange(3)-.2, [nvar, var, true_var], .4, label=f'VaR {q:.3%}'); ax[1].bar(np.arange(3)+.2, [nes, es, true_es], .4, label='ES')
    ax[1].set_xticks(range(3)); ax[1].set_xticklabels(labels); ax[1].legend(); ax[1].set_title('Normal badly underestimates tail risk; GPD tracks the truth')
    plt.tight_layout(); plt.show()
interact(pot_demo, nu=(2.2, 10.0, 0.1), N=(1000, 20000, 500), u_quantile=(0.85, 0.99, 0.01), q=(0.99, 0.9995, 0.0005), seed=(0, 30, 1));

interactive(children=(FloatSlider(value=0.3, description='xi', max=1.0, min=-0.5, step=0.02), FloatSlider(valu…

interactive(children=(FloatSlider(value=3.0, description='nu', max=10.0, min=2.2), IntSlider(value=5000, descr…

---
# PART E: MULTIVARIATE

---
## 22. Multivariate Normal $N(\boldsymbol\mu,\Sigma)$

### Meaning
A vector of returns $\mathbf R=(R_1,\dots,R_d)$ with mean vector $\boldsymbol\mu$ and covariance matrix $\Sigma$ ($\Sigma_{ij}=\operatorname{Cov}(R_i,R_j)$, $\Sigma_{ii}=\sigma_i^2$). Correlation $\rho_{ij}=\Sigma_{ij}/(\sigma_i\sigma_j)$.

### Formula
$$f(\mathbf x)=\frac1{(2\pi)^{d/2}|\Sigma|^{1/2}}\exp\Big(-\tfrac12(\mathbf x-\boldsymbol\mu)^\top\Sigma^{-1}(\mathbf x-\boldsymbol\mu)\Big).$$
Level sets of $f$ are **ellipses** (the Mahalanobis distance is constant); their tilt is set by $\rho$.

### Construction and derivation of moments
Write $\Sigma=LL^\top$ (Cholesky). If $\mathbf Z$ has iid $N(0,1)$ entries, then $\mathbf X=\boldsymbol\mu+L\mathbf Z$ is $N(\boldsymbol\mu,\Sigma)$ because
$$E[\mathbf X]=\boldsymbol\mu,\qquad\operatorname{Cov}(\mathbf X)=E[L\mathbf Z\mathbf Z^\top L^\top]=L\,I\,L^\top=\Sigma.$$
This is exactly how Monte Carlo generates correlated returns. For $2$ assets: $X_1=\mu_1+\sigma_1Z_1$, $X_2=\mu_2+\sigma_2(\rho Z_1+\sqrt{1-\rho^2}Z_2)$.
Any linear combination is Normal: $\mathbf w^\top\mathbf X\sim N(\mathbf w^\top\boldsymbol\mu,\ \mathbf w^\top\Sigma\mathbf w)$. Conditionals and marginals are Normal too: $X_2\mid X_1=x\sim N\big(\mu_2+\rho\frac{\sigma_2}{\sigma_1}(x-\mu_1),\ \sigma_2^2(1-\rho^2)\big)$.

### Portfolio variance (Markowitz)
$$\operatorname{Var}(\mathbf w^\top\mathbf R)=\mathbf w^\top\Sigma\mathbf w=\sum_i\sum_jw_iw_j\Sigma_{ij}.$$
For two assets: $w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\rho\sigma_1\sigma_2$. Diversification: lower $\rho$ means lower portfolio risk. Parametric VaR: $\mathrm{VaR}_q=-\mathbf w^\top\boldsymbol\mu+z_q\sqrt{\mathbf w^\top\Sigma\mathbf w}$.

### Why it underestimates joint crashes
Everything is described by means and covariances (a single $\rho$). The Normal has **zero tail dependence**: far out in the tails, extreme events in the two assets become independent. Real markets: when one asset crashes the others tend to crash simultaneously. A **multivariate Student-t** (or t-copula) with the same $\rho$ puts many more points in the joint corners. See the scatter below (compare the bottom-left corner).

In [24]:
def plot_mvn(sigma1=1.0, sigma2=1.0, rho=0.6, w1=0.5, nu=3.0, N=3000):
    rng = np.random.default_rng(8); S = np.array([[sigma1**2, rho*sigma1*sigma2], [rho*sigma1*sigma2, sigma2**2]])
    L = np.linalg.cholesky(S); Z = rng.standard_normal((N, 2)); X = Z @ L.T
    W = rng.chisquare(nu, N)/nu; T = (Z / np.sqrt(W)[:, None]) @ L.T * np.sqrt((nu-2)/nu)   # t with same covariance
    w = np.array([w1, 1-w1]); pv = w @ S @ w
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.5)); lim = 6
    ax[0].scatter(*X.T, s=4, alpha=.4); gx, gy = np.meshgrid(np.linspace(-lim, lim, 150), np.linspace(-lim, lim, 150))
    ax[0].contour(gx, gy, st.multivariate_normal([0, 0], S).pdf(np.dstack([gx, gy])), colors='r', levels=6)
    ax[0].set_title(f'Bivariate Normal (rho={rho:.2f})'); ax[1].scatter(*T.T, s=4, alpha=.4, color='C1')
    ax[1].set_title(f'Bivariate Student-t (nu={nu:.1f}), same cov: joint crashes!')
    for a in ax[:2]: a.set_xlim(-lim, lim); a.set_ylim(-lim, lim); a.set_aspect('equal')
    ws = np.linspace(0, 1, 101); pvs = [np.array([a, 1-a]) @ S @ np.array([a, 1-a]) for a in ws]
    ax[2].plot(ws, np.sqrt(pvs), 'k'); ax[2].plot(w1, np.sqrt(pv), 'ro'); ax[2].set_xlabel('weight in asset 1'); ax[2].set_ylabel('portfolio std dev')
    ax[2].set_title(f'Portfolio risk w^T S w: std={np.sqrt(pv):.3f}')
    thr = -2.5; jn = np.mean((X[:, 0] < thr) & (X[:, 1] < thr)); jt = np.mean((T[:, 0] < thr) & (T[:, 1] < thr))
    fig.suptitle(f'P(both < {thr}):  Normal = {jn:.4f}   Student-t = {jt:.4f}'); plt.tight_layout(); plt.show()
interact(plot_mvn, sigma1=(0.3, 2.0, 0.1), sigma2=(0.3, 2.0, 0.1), rho=(-0.95, 0.95, 0.05), w1=(0.0, 1.0, 0.05), nu=(2.5, 30.0, 0.5), N=(500, 8000, 500));

interactive(children=(FloatSlider(value=1.0, description='sigma1', max=2.0, min=0.3), FloatSlider(value=1.0, d…

---
# CHEAT SHEET: which distribution for what data?

| Data type | Distribution | Key parameters / notes |
|---|---|---|
| Yes/no event | Bernoulli($p$) | mean $p$, var $p(1-p)$ |
| Count out of $n$ trials | Binomial($n,p$) | mean $np$, var $np(1-p)$ |
| Trials until first success | Geometric($p$) | mean $1/p$, memoryless |
| Count of random events | Poisson($\lambda$) (Neg. Binomial if overdispersed) | mean = var = $\lambda$ |
| Time between events / constant-hazard default | Exponential($\lambda$) | mean $1/\lambda$, PD $=1-e^{-\lambda t}$ |
| Time with changing hazard | Weibull($k,\lambda$) | $k<1$ falling, $k>1$ rising hazard |
| Waiting time to $k$-th event, loss severity | Gamma($k,\theta$) | mean $k\theta$, var $k\theta^2$ |
| Prices (positive) | Lognormal | mean $e^{\mu+\sigma^2/2}$, median $e^\mu$ |
| Returns, central part | Normal | zero skew, kurtosis 3 |
| Returns incl. fat tails | Student-t, Laplace, stable | t: $\nu\approx3$-6 |
| Variance tests | Chi-square, F | $\chi^2_k$ mean $k$, var $2k$ |
| Recovery rates, probabilities | Beta | mean $\alpha/(\alpha+\beta)$ |
| Loss severities | Gamma, lognormal, Pareto | body vs tail |
| Extreme losses | GPD / GEV / Pareto | tail index $\xi=1/\alpha$ |
| Vector of returns | Multivariate Normal / t | $\mathbf w^\top\Sigma\mathbf w$ |

**Fat-tail ladder (thin to fat):** Uniform (bounded) < Normal < Laplace < Lognormal < Student-t($\nu$) < Pareto($\alpha$) < Cauchy < Levy.

**Key moment-existence rule:** For power-law tails $P(X>x)\sim x^{-\alpha}$ the $n$-th moment exists only if $n<\alpha$.

In [25]:
# Tail comparison: how likely is a "k-sigma" move under each model? (all scaled to variance 1)
def tail_compare(nu=4.0, kmax=10):
    k = np.linspace(0.5, kmax, 300)
    s_t = np.sqrt(nu/(nu-2)); s_l = np.sqrt(2)
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
    P = {'Normal': st.norm.sf(k), f'Student-t({nu:.1f})': st.t(nu).sf(k*s_t), 'Laplace': st.laplace(scale=1/s_l).sf(k)}
    for name, v in P.items(): ax[0].semilogy(k, v, label=name)
    ax[0].set_ylim(1e-16, 1); ax[0].set_xlabel('k (std devs)'); ax[0].set_ylabel('P(X > k sigma)'); ax[0].legend(); ax[0].set_title('Tail probabilities, all with unit variance')
    ratio = st.t(nu).sf(k*s_t)/st.norm.sf(k); ax[1].semilogy(k, ratio, 'k'); ax[1].set_title('How many times more likely is a k-sigma event under t than Normal?'); ax[1].set_xlabel('k')
    plt.tight_layout(); plt.show()
    for kk in [3, 5, 8]:
        print(f'k={kk}: Normal 1-in-{1/st.norm.sf(kk):,.0f}   t({nu:.1f}) 1-in-{1/st.t(nu).sf(kk*s_t):,.0f}')
interact(tail_compare, nu=(2.2, 30.0, 0.2), kmax=(4, 15, 1));

interactive(children=(FloatSlider(value=4.0, description='nu', max=30.0, min=2.2, step=0.2), IntSlider(value=1…

---
### End of notebook
**Suggested next steps:** (1) fit these distributions to real data (`scipy.stats.<dist>.fit`, QQ-plots with `st.probplot`); (2) backtest VaR/ES using Normal vs t vs GPD; (3) build copulas (Gaussian vs t) to model joint tail risk; (4) Monte Carlo pricing using the inverse-transform and Cholesky recipes derived above.